# 10-Fold Data Splitting Template

This notebook is a template for creating 10-fold cross-validation splits.

**Before running:**
1. Edit `config.json` with your data paths and column mappings
2. Set `objective_coverages` to exclude coverages you don't want in the objective function

**Steps:**
1. Load configuration and data
2. Explore all 6 coverage distributions
3. Analyze BI vs PD relationship
4. Remove zero exposure records
5. Calculate Pure Premium
6. Run simulations to find optimal seed
7. Apply best seed and validate
8. Save output

## 1. Setup and Imports

In [ ]:
# Standard imports
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import json

# Import functions from utils.py
from utils import (
    load_config,
    load_and_merge_data,
    show_zero_exposure_stats,
    calculate_pure_premium,
    calculate_overall_stats,
    print_overall_stats,
    run_simulations,
    get_best_seed,
    plot_objective_distribution,
    plot_all_coverage_histograms,
    plot_incurred_histograms,
    analyze_bi_pd_relationship,
    assign_folds,
    validate_folds,
    save_output
)

# Load configuration
config = load_config('config.json')

print("Configuration loaded!")
print(f"\nFile paths:")
print(f"  Main data: {config['file_paths']['main_data_path']}")
print(f"  Superpolicy: {config['file_paths']['superpolicy_path']}")
print(f"  Output: {config['file_paths']['output_path']}")
print(f"\nSimulation parameters:")
print(f"  N_SIMULATIONS: {config['n_simulations']}")
print(f"  N_FOLDS: {config['n_folds']}")
print(f"\nCoverages:")
print(f"  All: {config['all_coverages']}")
print(f"  For analysis: {config['analysis_coverages']}")
print(f"  For objective function: {config['objective_coverages']}")

## 2. Load and Merge Data

In [ ]:
# Load main data and superpolicy data, merge on join key
df = load_and_merge_data(config)

In [ ]:
# Quick look at the data
print(f"Shape: {df.shape}")
print(f"\nColumns: {list(df.columns)}")
df.head()

## 3. Explore All 6 Coverage Distributions

Visualize exposure and incurred loss distributions for all 6 coverages: BI, PD, PIP, MED, COLL, COMP

In [ ]:
# Plot histograms of all 6 EE columns
plot_all_coverage_histograms(df, config, coverages=config['all_coverages'])

In [ ]:
# Plot histograms of all 6 INCURRED columns
plot_incurred_histograms(df, config, coverages=config['all_coverages'])

### Coverage Notes

Based on the above analysis:
- **MED** coverage typically has 100% zero values (not available in many states)
- Coverages with high zero percentages may indicate optional coverages
- The `objective_coverages` in config.json excludes MED by default

**To modify which coverages are used in the objective function**, edit `config.json`:
```json
"coverages": {
    "objective_coverages": ["bi", "pd", "pip", "coll", "comp"]
}
```

## 4. Analyze BI vs PD Relationship

Determine if BI and PD should be combined or treated separately in the objective function.

In [ ]:
# Analyze BI vs PD relationship
analyze_bi_pd_relationship(df, config)

## 5. Zero Exposure Statistics

**Note:** All records are retained for fold assignment. Records with zero exposure for a coverage will have NaN Pure Premium for that coverage, but aggregate PP calculations handle this correctly.

In [ ]:
# Show zero exposure statistics (no records removed)
# All records will be assigned to folds
show_zero_exposure_stats(df, config)

In [ ]:
# Verify all records retained
print(f"Data shape (all records retained): {df.shape}")
df.head()

## 6. Calculate Pure Premium

In [ ]:
# Calculate Pure Premium for each coverage in analysis_coverages
df = calculate_pure_premium(df, config)

In [ ]:
# Calculate overall statistics (uses objective_coverages)
overall_stats = calculate_overall_stats(df, config)
print_overall_stats(overall_stats, config=config)

## 7. Run Simulations to Find Optimal Seed

In [ ]:
# Run simulations
# Uses objective_coverages from config (excludes MED)
results_df = run_simulations(df, overall_stats, config=config)

In [ ]:
# View simulation results
results_df.head(10)

## 8. View Objective Function Distribution

In [ ]:
# Plot histogram of objective function values
plot_objective_distribution(results_df)

In [ ]:
# Find the best seed
best_seed, best_obj = get_best_seed(results_df)

## 9. Apply Best Seed and Assign Folds

In [ ]:
# Assign folds using the best seed
df = assign_folds(df, best_seed, config)

print(f"\nFold distribution:")
print(df['fold'].value_counts().sort_index())

## 10. Validate Fold Assignment

In [ ]:
# Validate the fold assignment quality
validate_folds(df, overall_stats, config)

## 11. Save Output

In [ ]:
# Save fold assignments and simulation results
df_output = save_output(df, simulation_results=results_df, config=config)

In [ ]:
# Preview final output
print(f"\nOutput preview:")
df_output.head(10)

## Done!

The 10-fold assignments have been saved. You can now merge this output back to your original data using the join key.

In [ ]:
# Example: How to merge back to original data
# original_data = pd.read_parquet("your_original_data.parquet")
# merged = pd.merge(original_data, df_output[['vin_date', 'fold']], on='vin_date', how='left')